# Regression trees: local means, pruning, and residual evidence

**Lecture 14 · Notebook 01 · CMOR 438 / INDE 577**

A regression tree divides feature space into rectangles and predicts one local
target mean per leaf. We will derive squared-error split reduction, fit the course
implementation, tune structural complexity, and test the failure modes that matter
when a discontinuous model becomes a reusable software component.


## How to use this notebook

Complete Notebook 00 first. Run **core** cells in order with the **Rice DSM**
kernel, narrating what each plot or assertion is evidence for. The **practice** and
**extension** prompts become project work. Keep reusable tree logic in `src/` and
import it here; never copy a second implementation into the notebook.


## Learning objectives

You will be able to:

1. derive squared-error reduction and the leaf-mean prediction;
2. interpret a regression tree as an adaptive histogram;
3. select depth, leaf size, and pruning strength without test leakage;
4. diagnose residual structure, instability, discontinuities, and extrapolation;
5. compare MAE and RMSE with a mean baseline; and
6. define production tests for data schema and threshold behavior.


## Why this matters

Regression trees capture nonlinearities and interactions without specifying a
polynomial or scaling the features. They can also be brittle: predictions jump at
thresholds, remain constant inside leaves, and cannot extrapolate beyond observed
leaf means. These limitations make a single tree an excellent transparent
baseline—and motivate random forests and boosting in the next lecture.

**Professional practice:** the goal is not to draw the prettiest tree. The goal is
to produce a reproducible prediction artifact with declared data, loss, validation
boundary, latency, schema, and monitoring expectations.


In [ ]:
# Standard-library imports handle paths, the read-only database, and diagnostics.
from pathlib import Path
import sqlite3
import sys

# Scientific libraries support numerical work, tables, and visual evidence.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Reuse the maintained package implementation instead of copying it into a cell.
from rice_dsm.ml.trees import DecisionTreeRegressor, mean_squared_impurity, plot_tree
from rice_dsm.paths import course_database_path, find_project_root

# Resolve project resources without hard-coded macOS, Linux, or Windows paths.
PROJECT_ROOT = find_project_root(Path.cwd())
DATABASE_PATH = course_database_path(PROJECT_ROOT)
print("Python:", sys.executable)
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve()
assert DATABASE_PATH.is_file()


## 1. Query a real regression dataset and audit its meaning

The diabetes dataset contains ten baseline variables for 442 patients and a
quantitative disease-progression response measured one year later. Several feature
meanings are incompletely documented; this is a limitation, not a detail to hide.
The bundled values are not a license for clinical deployment.

We query the versioned SQLite database using the standard-library
[`sqlite3` API](https://docs.python.org/3/library/sqlite3.html), then inspect the
schema, target distribution, duplicates, and plausible ranges before modeling.


In [ ]:
# Read-only mode prevents exploratory code from mutating the course database.
database_uri = f"file:{DATABASE_PATH.as_posix()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    diabetes = pd.read_sql_query(
        "SELECT * FROM diabetes_observations ORDER BY observation_id",
        connection,
    )

# Declare one ordered schema used consistently by fit, predict, and visualization.
feature_names = ["age", "sex", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]
target_name = "disease_progression"
# Inspect both individual records and distribution summaries before modeling.
display(diabetes.head())
display(diabetes[[*feature_names, target_name]].describe().T)

# Executable checks make data assumptions fail close to the source.
assert diabetes.shape == (442, 12)
assert diabetes[feature_names + [target_name]].isna().sum().sum() == 0
assert diabetes["observation_id"].is_unique
assert diabetes[target_name].nunique() > 100


In [ ]:
# Explore target shape and two marginal relationships before choosing a model.
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=diabetes, x=target_name, bins=25, ax=axes[0])
sns.scatterplot(data=diabetes, x="bmi", y=target_name, alpha=0.7, ax=axes[1])
sns.scatterplot(data=diabetes, x="s5", y=target_name, alpha=0.7, ax=axes[2])
axes[0].set_title("Target distribution")
axes[1].set_title("BMI has signal and substantial variation")
axes[2].set_title("A second feature shows a nonlinear-looking association")
figure.tight_layout()


## 2. Squared-error reduction creates local means

Here impurity means **how much unresolved response variation remains inside a
node**. For targets in node $S$, the constant minimizing squared error is the node
mean:

$$\hat c_S=\arg\min_c\sum_{i\in S}(y_i-c)^2=\bar y_S.$$

A candidate split $x_j\le t$ is scored by

$$\Delta\mathrm{SSE}=\mathrm{SSE}(S)-\mathrm{SSE}(S_L)-\mathrm{SSE}(S_R),$$

where $\mathrm{SSE}(A)=\sum_{i\in A}(y_i-\bar y_A)^2$. Squaring prevents positive
and negative residuals from canceling and penalizes large misses more strongly.
It also makes the mean the unique best constant. Dividing SSE by node size gives
mean-squared impurity, the within-node variance using divisor $n$.

The reduction has a variance-decomposition interpretation: parent variation equals
variation left within the two children plus variation explained by separating their
means. A useful question makes child responses more homogeneous. Child SSE is
added, equivalently weighted by child size when MSE is used; an unweighted average
would overvalue tiny leaves. The greedy algorithm chooses the largest immediate
reduction, then repeats inside each child. The fitted function is piecewise
constant even when the underlying response is smooth.


In [ ]:
# Verify visually why a regression leaf predicts the mean under squared error.
leaf_targets = np.array([2.0, 3.0, 4.0, 11.0])
candidate_constants = np.linspace(0.0, 13.0, 400)
squared_loss = np.array(
    [np.sum((leaf_targets - constant) ** 2) for constant in candidate_constants]
)
absolute_loss = np.array(
    [np.sum(np.abs(leaf_targets - constant)) for constant in candidate_constants]
)

# Put the losses side by side to connect the objective with its leaf statistic.
figure, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].plot(candidate_constants, squared_loss, color="#00205b")
axes[0].axvline(leaf_targets.mean(), color="#c8102e", linestyle="--", label="mean")
axes[0].set(
    title="Squared loss is minimized by the mean",
    xlabel="leaf prediction c",
    ylabel="total loss",
)
axes[0].legend(frameon=False)
axes[1].plot(candidate_constants, absolute_loss, color="#00205b")
axes[1].axvline(np.median(leaf_targets), color="#f2a900", linestyle="--", label="median interval")
axes[1].set(
    title="Absolute loss is minimized by a median",
    xlabel="leaf prediction c",
    ylabel="total loss",
)
axes[1].legend(frameon=False)
figure.tight_layout()

assert np.isclose(
    candidate_constants[np.argmin(squared_loss)], leaf_targets.mean(), atol=0.02
)


### Worked example: derive the best split on six observations

The targets below have a visible jump. Enumerate midpoint thresholds and verify
that minimizing child SSE is equivalent to maximizing SSE reduction.


In [ ]:
# Enumerate midpoint thresholds and compute each pair of local means.
toy_x = np.array([1.0, 2.0, 3.0, 5.0, 8.0, 9.0])
toy_y = np.array([2.0, 3.0, 2.0, 8.0, 9.0, 10.0])
parent_sse = len(toy_y) * mean_squared_impurity(toy_y)
# Keep every candidate's means and losses for a transparent audit table.
rows = []
for threshold in (toy_x[:-1] + toy_x[1:]) / 2:
    left = toy_y[toy_x <= threshold]
    right = toy_y[toy_x > threshold]
    child_sse = (
        len(left) * mean_squared_impurity(left)
        + len(right) * mean_squared_impurity(right)
    )
    rows.append(
        {
            "threshold": threshold,
            "left_mean": left.mean(),
            "right_mean": right.mean(),
            "child_sse": child_sse,
            "sse_reduction": parent_sse - child_sse,
        }
    )

split_table = pd.DataFrame(rows)
display(split_table.style.highlight_max(subset="sse_reduction", color="#d9ead3"))
best_toy_threshold = split_table.loc[split_table["child_sse"].idxmin(), "threshold"]
assert best_toy_threshold == 4.0
assert np.isclose(toy_y.mean(), 34 / 6)


In [ ]:
# Plot the stump as a discontinuous piecewise-constant prediction function.
toy_tree = DecisionTreeRegressor(max_depth=1).fit(toy_x[:, None], toy_y)
grid = np.linspace(0.5, 9.5, 400)
figure, axis = plt.subplots(figsize=(9, 4.5))
axis.scatter(toy_x, toy_y, color="#00205b", s=60, label="observations")
axis.step(grid, toy_tree.predict(grid[:, None]), where="post", color="#c8102e", label="tree prediction")
axis.axvline(toy_tree.tree_.threshold, color="black", linestyle="--", label="chosen threshold")
axis.set(xlabel="feature", ylabel="response", title="A regression stump returns one mean per region")
axis.legend()
figure.tight_layout()
assert toy_tree.tree_.threshold == 4.0


## 3. Separate estimation, selection, and final evaluation

A random split is an instructional baseline, not automatically the scientifically
correct validation design. Repeated patients, sites, batches, or time require
grouped or temporal splits. Here each row is a distinct patient and no timestamp
is available, so we state the limitation and use fixed train/validation/test roles.

No standardization is needed for the tree, because thresholds depend on order.
That invariance does not eliminate leakage from imputation, feature selection, or
target-informed transformations.


In [ ]:
# Create the untouched test set before considering any model complexity.
from sklearn.model_selection import train_test_split

# Preserve the declared feature order when moving from a table to NumPy arrays.
X = diabetes[feature_names].to_numpy(dtype=float)
y = diabetes[target_name].to_numpy(dtype=float)
# The test targets remain sealed until the validation choice is frozen.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=438
)
# Training estimates splits; validation chooses depth and minimum leaf size.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=438
)

assert len(X_train) + len(X_validation) + len(X_test) == 442
assert set(map(tuple, X_train)).isdisjoint(set(map(tuple, X_test)))
print(len(X_train), len(X_validation), len(X_test))


### Worked example: inspect one shallow regression tree

Exported rules make the computation auditable. A leaf mean describes this fitted
sample; it is not a treatment recommendation and does not explain why progression
occurred.


In [ ]:
# Keep this first fitted tree shallow enough to verify every leaf mean manually.
course_tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20).fit(
    X_train, y_train
)
# Text rules support tests and logs; the next cell exposes their hierarchy.
print(course_tree.export_text(feature_names))
print("depth:", course_tree.get_depth(), "leaves:", course_tree.get_n_leaves())

assert course_tree.get_depth() <= 2
assert course_tree.get_n_leaves() <= 4
assert np.isclose(course_tree.feature_importances_.sum(), 1.0)
assert np.isfinite(course_tree.predict(X_validation)).all()


In [ ]:
# Visualize the same fitted regression tree and its local-mean evidence.
figure, axis = plt.subplots(figsize=(12, 6))
plot_tree(course_tree, feature_names, ax=axis, precision=2)
axis.set_title(
    "Regression tree: questions lead to local conditional means",
    color="#00205b",
    weight="bold",
)
figure.tight_layout()


In [ ]:
# Validation selects structural regularization; test labels remain untouched.
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Search a declared grid and record both predictive error and tree size.
candidates = []
for max_depth in (1, 2, 3, 4, 6, 10):
    for min_leaf in (5, 10, 20, 35):
        model = DecisionTreeRegressor(
            max_depth=max_depth, min_samples_leaf=min_leaf
        ).fit(X_train, y_train)
        validation_prediction = model.predict(X_validation)
        candidates.append(
            {
                "max_depth": max_depth,
                "min_leaf": min_leaf,
                "validation_mae": mean_absolute_error(y_validation, validation_prediction),
                "validation_rmse": mean_squared_error(y_validation, validation_prediction) ** 0.5,
                "leaves": model.get_n_leaves(),
            }
        )

validation_results = pd.DataFrame(candidates).sort_values(
    ["validation_rmse", "leaves"]
)
best = validation_results.iloc[0]
display(validation_results.head(10))
assert best["validation_rmse"] > 0


## 4. Baseline, frozen model, and held-out residual evidence

MAE is the average absolute miss in target units. RMSE emphasizes large misses.
Neither number is meaningful without a baseline and response context. Once the
structural choice is frozen, refit on training plus validation and evaluate the
test set once.


In [ ]:
# Freeze the selected structure and compare it with a mean-only baseline.
from sklearn.dummy import DummyRegressor

final_tree = DecisionTreeRegressor(
    max_depth=int(best["max_depth"]),
    min_samples_leaf=int(best["min_leaf"]),
).fit(X_train_full, y_train_full)
baseline = DummyRegressor(strategy="mean").fit(X_train_full, y_train_full)

# Compare on the same sealed test observations and in the same target units.
predictions = final_tree.predict(X_test)
baseline_predictions = baseline.predict(X_test)
metrics = pd.DataFrame(
    {
        "model": ["mean baseline", "decision tree"],
        "MAE": [
            mean_absolute_error(y_test, baseline_predictions),
            mean_absolute_error(y_test, predictions),
        ],
        "RMSE": [
            mean_squared_error(y_test, baseline_predictions) ** 0.5,
            mean_squared_error(y_test, predictions) ** 0.5,
        ],
    }
)
display(metrics)
assert len(predictions) == len(y_test)
assert np.isfinite(metrics[["MAE", "RMSE"]].to_numpy()).all()


In [ ]:
# Residual plots test structure that scalar metrics hide.
residuals = y_test - predictions
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(predictions, y_test, alpha=0.75)
bounds = [min(predictions.min(), y_test.min()), max(predictions.max(), y_test.max())]
axes[0].plot(bounds, bounds, linestyle="--", color="black")
axes[0].set(xlabel="predicted progression", ylabel="observed progression", title="Prediction versus observation")
axes[1].scatter(predictions, residuals, alpha=0.75)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="predicted progression", ylabel="observed - predicted", title="Held-out residual structure")
figure.tight_layout()
assert np.isclose(residuals, y_test - predictions).all()


## 5. Pruning and the bias-variance tradeoff

Deep trees reduce training error by creating small regions, increasing variance.
Cost-complexity pruning adds a penalty for each terminal region. We use scikit-
learn's weakest-link pruning path as a trusted optimized reference and select the
penalty on validation data. A one-standard-error rule is often preferable to
mechanically choosing the smallest observed validation error.


In [ ]:
# Trace the trusted weakest-link pruning sequence on training data only.
from sklearn.tree import DecisionTreeRegressor as SklearnTreeRegressor

base_tree = SklearnTreeRegressor(random_state=438, min_samples_leaf=5)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
pruning_rows = []
# Validation selects alpha while training error reveals the expected optimism.
for alpha in np.unique(path.ccp_alphas):
    model = SklearnTreeRegressor(
        random_state=438, min_samples_leaf=5, ccp_alpha=float(alpha)
    ).fit(X_train, y_train)
    pruning_rows.append(
        {
            "alpha": alpha,
            "leaves": model.get_n_leaves(),
            "depth": model.get_depth(),
            "validation_rmse": mean_squared_error(
                y_validation, model.predict(X_validation)
            ) ** 0.5,
        }
    )

pruning = pd.DataFrame(pruning_rows)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].step(pruning["alpha"], pruning["leaves"], where="post")
axes[0].set(xlabel="cost-complexity alpha", ylabel="leaf count", title="Complexity path")
axes[1].plot(pruning["alpha"], pruning["validation_rmse"], marker=".")
axes[1].set(xlabel="cost-complexity alpha", ylabel="validation RMSE", title="Selection evidence")
figure.tight_layout()
assert pruning.iloc[-1]["leaves"] == 1


## 6. Discontinuity, instability, and extrapolation are testable

A threshold creates a jump: nearly identical inputs can enter different leaves.
Bootstrap refits reveal that root variables and cut points can change. Outside the
training range, a tree does not continue a trend; it returns an existing leaf mean.
These properties should appear in documentation and monitoring plans.


In [ ]:
# Check the exact <= boundary contract at the root.
root = final_tree.tree_
assert root.feature_index is not None and root.threshold is not None
reference = X_test[0].copy()
below, equal, above = reference.copy(), reference.copy(), reference.copy()
below[root.feature_index] = np.nextafter(root.threshold, -np.inf)
equal[root.feature_index] = root.threshold
above[root.feature_index] = np.nextafter(root.threshold, np.inf)
boundary_predictions = final_tree.predict([below, equal, above])
print(feature_names[root.feature_index], root.threshold, boundary_predictions)
assert final_tree.apply([below])[0] == final_tree.apply([equal])[0]


In [ ]:
# Demonstrate that the model cannot extrapolate beyond learned leaf means.
extreme_low = X_train_full.min(axis=0) - 10 * X_train_full.std(axis=0)
extreme_high = X_train_full.max(axis=0) + 10 * X_train_full.std(axis=0)
extreme_predictions = final_tree.predict([extreme_low, extreme_high])
training_leaf_predictions = final_tree.predict(X_train_full)
print("extreme predictions:", extreme_predictions)
print("training prediction range:", training_leaf_predictions.min(), training_leaf_predictions.max())
assert extreme_predictions.min() >= training_leaf_predictions.min()
assert extreme_predictions.max() <= training_leaf_predictions.max()


In [ ]:
# Bootstrap root questions to measure structural instability.
rng = np.random.default_rng(438)
roots = []
for _ in range(80):
    sample = rng.integers(0, len(X_train_full), size=len(X_train_full))
    model = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20).fit(
        X_train_full[sample], y_train_full[sample]
    )
    roots.append(
        {
            "feature": feature_names[model.tree_.feature_index],
            "threshold": model.tree_.threshold,
        }
    )

root_fits = pd.DataFrame(roots)
display(root_fits.groupby("feature").agg(count=("threshold", "size"), threshold_std=("threshold", "std")).sort_values("count", ascending=False))
assert len(root_fits) == 80


## 7. Interpretation and operational evidence

A path explains which conditions in this fitted tree produced a prediction. It does
not explain the patient's biology or an intervention. Impurity importance can
over-credit variables with many possible split points and behave unpredictably
with correlated inputs. Pair it with held-out permutation evidence, resampling,
and domain review.

In production, log the model version, schema version, missingness, out-of-range
rates, reached-leaf distribution, latency, and eventual residuals when outcomes
arrive. A sudden shift into rarely visited leaves is actionable even before labels
are available.


In [ ]:
# Compare built-in split-credit importance with held-out predictive disruption.
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor as TrustedTreeRegressor

# The optimized reference estimator supplies a standard permutation interface.
trusted = TrustedTreeRegressor(
    max_depth=int(best["max_depth"]),
    min_samples_leaf=int(best["min_leaf"]),
    random_state=438,
).fit(X_train_full, y_train_full)
# Shuffle one test feature at a time; repeat to expose sampling variability.
permutation = permutation_importance(
    trusted, X_test, y_test, scoring="neg_mean_squared_error", n_repeats=20, random_state=438
)
# Keep the two meanings side by side--they need not rank variables identically.
importance = pd.DataFrame(
    {
        "feature": feature_names,
        "impurity_importance": trusted.feature_importances_,
        "permutation_mean": permutation.importances_mean,
        "permutation_std": permutation.importances_std,
    }
).sort_values("permutation_mean", ascending=False)
display(importance)
assert set(importance["feature"]) == set(feature_names)


## 8. Professional project handoff

Create or update your project from the [companion student template](https://github.com/RandyRDavila/Data-Science-Machine-Learning-Project-Template).
From a clean template `main`, run:

```text
git switch main
git pull --ff-only
git switch -c YOUR-GITHUB-USERNAME/regression-tree
uv sync --locked
```

Place reusable code in
`src/dsm_project/algorithms/student/trees.py`, analytic and failure tests in
`tests/test_student_trees.py`, register the estimator in
`src/dsm_project/algorithms/registry.py`, put small lawful CI data in
`data/fixtures/`, and keep private or large raw data in ignored `data/local/`.
The applied notebook imports the package and tells a question-led story; it does
not become a second implementation.

Work issue-first: state the response, prediction unit, split policy, and baseline;
write a leaf-mean and threshold test; implement the estimator; add the real-data
analysis; then run:

```text
uv run ruff check src tests scripts
uv run pytest -q
git status --short
git add src tests notebooks docs data/fixtures
git commit -m "Add regression-tree case study"
git push -u origin YOUR-GITHUB-USERNAME/regression-tree
```

Your dataset card should record provenance, license, collection process,
observational unit, target timing, known gaps, sensitive attributes, and prohibited
uses. Open a focused pull request that links the issue and reports baseline, split
policy, validation evidence, held-out metrics, residual diagnostics, threshold
tests, limitations, and one failure you investigated.


### Student contribution test plan

| Test | Contract protected |
| --- | --- |
| a two-point leaf predicts its arithmetic mean | leaf optimization |
| six-row fixture selects threshold $4$ | SSE split correctness |
| equality at the threshold routes left | boundary semantics |
| predictions stay within observed leaf means | no-extrapolation behavior |
| invalid shape, NaN, and pre-fit prediction fail clearly | user-facing failures |
| fixed configuration agrees with scikit-learn | differential behavior |
| clean-kernel notebook execution | package/notebook boundary |


In [ ]:
# This smoke test demonstrates the package boundary students should reproduce.
# In the template repository, replace the next import with:
# from dsm_project.algorithms.student.trees import DecisionTreeRegressor
from rice_dsm.ml.trees import DecisionTreeRegressor as CourseRegressor

smoke_regressor = CourseRegressor(max_depth=1).fit(
    [[0.0], [1.0], [2.0], [3.0]],
    [1.0, 1.0, 5.0, 7.0],
)
assert np.allclose(smoke_regressor.predict([[0.5], [2.5]]), [1.0, 6.0])
print("The notebook imported tested package code; it did not redefine the tree.")


## Professional practice and common failure modes

- **Tiny leaves:** low training SSE but noisy conditional-mean estimates.
- **Test-set tuning:** selecting depth or alpha on final outcomes.
- **Unseen-range confidence:** forgetting that trees do not extrapolate.
- **Threshold brittleness:** ignoring equality, rounding, and unit conversion.
- **Random-split mismatch:** mixing patients, sites, batches, or future records.
- **Importance overreach:** treating impurity reduction as causality.
- **Silent schema changes:** reordered columns or altered measurement units.


## Debugging clinic

When a regression-tree prediction is surprising, reproduce the path manually.
Confirm feature order and units, inspect every `<=` comparison, print leaf sample
count and target mean, and compare the result with a tiny analytic fixture. If
validation performance changes unexpectedly, verify random seeds and that no
preprocessing or feature selection saw test targets.


## Guided practice

Fit two trees with the same depth but different `min_samples_leaf`. Compare leaf
counts, validation RMSE, and prediction ranges. Explain the change as a statistical
regularization effect rather than merely an API setting.


In [ ]:
# PRACTICE: compare two leaf-size constraints on the same fixed split.
small_leaf = DecisionTreeRegressor(max_depth=5, min_samples_leaf=3).fit(X_train, y_train)
stable_leaf = DecisionTreeRegressor(max_depth=5, min_samples_leaf=25).fit(X_train, y_train)
comparison = pd.DataFrame(
    {
        "model": ["small leaves", "larger leaves"],
        "leaves": [small_leaf.get_n_leaves(), stable_leaf.get_n_leaves()],
        "validation_rmse": [
            mean_squared_error(y_validation, small_leaf.predict(X_validation)) ** 0.5,
            mean_squared_error(y_validation, stable_leaf.predict(X_validation)) ** 0.5,
        ],
    }
)
display(comparison)
assert stable_leaf.get_n_leaves() <= small_leaf.get_n_leaves()
# TODO: explain which model you would advance and what uncertainty remains.


## Independent practice

Add a regression-tree analysis to your project. Use a real question and dataset,
implement or extend your package API, write hand-computable tests, establish a
mean/median baseline, choose complexity without test leakage, and include residual,
stability, threshold, and extrapolation evidence. Submit the work through an issue,
branch, tested pull request, and peer review.


## Extension: uncertainty and distribution shift

Use repeated or nested cross-validation to estimate uncertainty in the performance
difference between a pruned tree and baseline. Then design a monitoring report for
feature ranges, leaf occupancy, and delayed residuals. Explain which alerts can be
computed without labels and which require outcomes to arrive.


## Retrieval practice

1. Why is the leaf mean the squared-error minimizer?
2. Why does a tree prediction jump at a threshold?
3. What distinct roles do depth, leaf size, and pruning alpha play?
4. Why can a regression tree not extrapolate a trend?
5. What residual pattern would make you question the partition?


## Takeaway and further reading

A regression tree is an adaptive, piecewise-constant estimator built by greedy SSE
reduction. Structural regularization and pruning trade training fit for stability.
Professional evidence includes a baseline, frozen validation boundary, residuals,
threshold tests, extrapolation limits, resampling, and schema-aware monitoring.

Further reading: [scikit-learn tree documentation](https://scikit-learn.org/stable/modules/tree.html),
[Python `dataclasses`](https://docs.python.org/3/library/dataclasses.html), and
Breiman, Friedman, Olshen, and Stone, *Classification and Regression Trees* (1984).
